Change to project root directory

In [1]:
from pathlib import Path

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / "pyproject.toml").exists():
    project_root = project_root.parent

%cd {project_root}

/hfm/songlin/psi


In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
import os
os.environ["FORCE_QWENVL_VIDEO_READER"] = "decord"  # decord2 installs as `decord`; torchcodec needs FFmpeg which is unavailable

from transformers import AutoProcessor, AutoModelForVision2Seq
from qwen_vl_utils import process_vision_info
import warnings
warnings.filterwarnings("ignore", category=FutureWarning, module="transformers")
from transformers import Qwen3VLForConditionalGeneration

/hfm/songlin/psi/.venv-psi/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
model_path = "Qwen/Qwen3-VL-2B-Instruct" 
processor = AutoProcessor.from_pretrained(model_path)

In [5]:
# AutoModelForVision2Seq.from_pretrained
model, output_loading_info = Qwen3VLForConditionalGeneration.from_pretrained(
    model_path, 
    dtype="auto", 
    device_map="auto", 
    output_loading_info=True
)

In [6]:
# Messages containing a video url(or a local path) and a text query
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "video",
                "video": "https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen2-VL/space_woaudio.mp4",
            },
            {"type": "text", "text": "Describe this video."},
        ],
    }
]

# Preparation for inference
inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt"
)
inputs = inputs.to(model.device)

# Inference: Generation of the output
generated_ids = model.generate(**inputs, max_new_tokens=128)
generated_ids_trimmed = [
    out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]
output_text = processor.batch_decode(
    generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
)
print(output_text)

['The video is a documentary-style presentation that explores the International Space Station (ISS) and the activities of astronauts aboard it. It begins with a man in a control room, who is likely a mission control operator, speaking to the camera. He is standing in front of a large screen displaying a map of the Earth, with various satellite signals and data streams visible. The man appears to be explaining something to the audience, possibly about the operations or mission control activities related to the ISS.\n\nThe video then transitions to a series of scenes that showcase the interior of the ISS. We see astronauts in the space station, working on various tasks, such as']
